# Notebook 04 — Insights & Visualisations

**Goal:** Produce all interactive Plotly charts and derive actionable personalization recommendations for each listener segment.

**Research question:** *What user segments exist based on listening diversity patterns, and how can streaming platforms use these insights to optimise personalization strategies for different listener types?*

**Inputs** (from `data/processed/`):
- `user_features.parquet`
- `cluster_labels.parquet`
- `scrobbles_updated.parquet`
- `artist_genres.parquet`
- `profiles.parquet`

**Outputs** (in `outputs/figures/`):
- `umap_clusters.html` — 2D UMAP scatter
- `cluster_heatmap.html` — feature profile heatmap
- `radar_chart.html` — radar comparison of segments
- `feature_importance.html` — discriminating features
- `genre_distribution.html` — genre breakdown per segment
- `temporal_heatmap_cluster_N.html` — listening patterns per segment

In [ ]:
import sys
import logging
from pathlib import Path

sys.path.insert(0, str(Path.cwd().parent))
logging.basicConfig(level=logging.INFO)

import pandas as pd
import numpy as np
import plotly.io as pio

pio.renderers.default = 'notebook'
Path('../outputs/figures').mkdir(parents=True, exist_ok=True)

In [ ]:
from pathlib import Path

feature_matrix = pd.read_parquet('../data/processed/user_features.parquet')
labels_df      = pd.read_parquet('../data/processed/cluster_labels.parquet')
artist_genres  = pd.read_parquet('../data/processed/artist_genres.parquet')
profiles       = pd.read_parquet('../data/processed/profiles.parquet')

labels       = labels_df['cluster'].values
cluster_names = dict(zip(labels_df['cluster'], labels_df['cluster_name']))
userids      = labels_df['userid'].tolist()

# Load scrobbles — prefer updated, but fall back to baseline if userids don't
# match cluster labels (happens when scrobbles_updated.parquet was written from
# a partial API run that used real usernames instead of the anonymised dataset ids).
_upd = Path('../data/processed/scrobbles_updated.parquet')
_base = Path('../data/processed/scrobbles_baseline.parquet')

def _load_scrobbles(path):
    return pd.read_parquet(path)

scrobbles = _load_scrobbles(_upd) if _upd.exists() else _load_scrobbles(_base)

# Sanity check: if fewer than 50% of cluster userids appear in the scrobbles
# file, the updated file was written from API data with non-anonymised userids.
uid_set = set(scrobbles['userid'].astype(str).str.strip())
label_uid_set = set(str(u).strip() for u in userids)
overlap = len(uid_set & label_uid_set)
overlap_pct = overlap / max(len(label_uid_set), 1)

if overlap_pct < 0.5 and _base.exists():
    print(f'scrobbles_updated.parquet only covers {overlap_pct:.0%} of cluster users — '
          f'switching to scrobbles_baseline.parquet')
    scrobbles = _load_scrobbles(_base)
    overlap = len(set(scrobbles['userid'].astype(str).str.strip()) & label_uid_set)

print(f'Scrobbles: {len(scrobbles):,} rows | Users in scrobbles matching clusters: {overlap}')
print(f'Users: {len(userids)} | Segments: {labels_df["cluster"].nunique()}')
labels_df['cluster_name'].value_counts()


In [ ]:
FINAL_PERSONA_NAMES = {
    0: 'The Casual Listener',
    1: 'The Loyalist',
    2: 'The Binge Listener',
    3: 'The Focused Listener',
    4: 'The Discoverer',
}

# Override auto-generated names — all visualizations below use cluster_names
cluster_names = FINAL_PERSONA_NAMES
labels_df['cluster_name'] = labels_df['cluster'].map(FINAL_PERSONA_NAMES)

print("Final persona mapping:")
for cid, name in FINAL_PERSONA_NAMES.items():
    n = (labels_df['cluster'] == cid).sum()
    pct = n / len(labels_df) * 100
    print(f"  Cluster {cid}: {name} ({n} users, {pct:.1f}%)")

## 1. UMAP Cluster Scatter

In [ ]:
from src.visualization.plots import plot_umap_clusters, save_figure

umap_coords = labels_df[['umap_x', 'umap_y']].values

fig_umap = plot_umap_clusters(
    umap_coords=umap_coords,
    labels=labels,
    cluster_names=cluster_names,
    userids=userids,
)
save_figure(fig_umap, '../outputs/figures/umap_clusters')
fig_umap.show()

## 2. Cluster Feature Heatmap

Z-scored means — red = above average, blue = below average.

In [ ]:
from src.visualization.plots import plot_cluster_heatmap
from src.clustering.evaluation import summarise_clusters

# Focus on the most interpretable features for the heatmap
key_features = [
    'artist_entropy', 'genre_entropy', 'unique_artists', 'artist_concentration_20',
    'track_replay_rate', 'novelty_ratio', 'discovery_velocity_30d',
    'avg_tracks_per_session', 'avg_session_length_min', 'weekend_ratio',
    'temporal_hour_entropy', 'morning_ratio', 'evening_ratio',
    'mean_energy', 'mean_valence', 'mean_danceability', 'mean_acousticness',
]
key_features = [f for f in key_features if f in feature_matrix.columns]

fig_heatmap = plot_cluster_heatmap(
    feature_matrix=feature_matrix,
    labels=labels,
    features=key_features,
    cluster_names=cluster_names,
)
save_figure(fig_heatmap, '../outputs/figures/cluster_heatmap')
fig_heatmap.show()

## 3. Radar Chart — Segment Profiles

In [ ]:
from src.visualization.plots import plot_cluster_radar

cluster_summary = summarise_clusters(feature_matrix, labels)

radar_features = [
    'artist_entropy', 'genre_entropy', 'novelty_ratio',
    'track_replay_rate', 'avg_tracks_per_session', 'temporal_hour_entropy',
]
radar_features = [f for f in radar_features if f in feature_matrix.columns]

fig_radar = plot_cluster_radar(
    cluster_summary=cluster_summary,
    features=radar_features,
    cluster_names=cluster_names,
)
save_figure(fig_radar, '../outputs/figures/radar_chart')
fig_radar.show()

## 4. Feature Importance

In [ ]:
from src.clustering.evaluation import feature_importance
from src.visualization.plots import plot_feature_importance

imp_df = feature_importance(feature_matrix, labels)

fig_imp = plot_feature_importance(imp_df, top_n=20)
save_figure(fig_imp, '../outputs/figures/feature_importance')
fig_imp.show()

## 5. Genre Distribution by Segment

In [ ]:
# Genre data diagnostic
from src.visualization.plots import _uid_str

print(f"artist_genres shape: {artist_genres.shape}")
sample = artist_genres['genres'].dropna().head(3)
print(f"genres dtype sample (type={type(sample.iloc[0]).__name__}): {sample.iloc[0]}")

has_genre = artist_genres['genres'].apply(lambda x: len(x) > 0 if hasattr(x, '__len__') else bool(x))
print(f"Artists with ≥1 genre: {has_genre.sum()} / {len(artist_genres)}")

# Check userid mapping coverage per cluster
sc_check = scrobbles.copy()
sc_check['artist_name_normalized'] = sc_check['artist_name'].str.strip().str.lower()
user_cluster_check = {_uid_str(uid): int(cid) for uid, cid in zip(userids, labels)}
sc_check['cluster'] = sc_check['userid'].apply(_uid_str).map(user_cluster_check)

print(f"\nScrobble userid dtype:   {scrobbles['userid'].dtype}")
print(f"labels_df userid dtype:  {labels_df['userid'].dtype}")
print(f"Sample scrobble uid:     {repr(scrobbles['userid'].iloc[0])}")
print(f"Sample labels_df uid:    {repr(labels_df['userid'].iloc[0])}")
print(f"After _uid_str — scrobble: {repr(_uid_str(scrobbles['userid'].iloc[0]))}")
print(f"After _uid_str — label:    {repr(_uid_str(labels_df['userid'].iloc[0]))}")

print("\nCluster distribution in scrobbles (NaN = unmapped users):")
print(sc_check['cluster'].value_counts(dropna=False).sort_index())

In [ ]:
from src.visualization.plots import plot_genre_distribution
import numpy as np, json as _json

# ── Build genre × segment percentage table ───────────────────────────────────
from src.visualization.plots import _uid_str
sc = scrobbles.copy()
sc['artist_name_normalized'] = sc['artist_name'].str.strip().str.lower()
user_cluster_map = {_uid_str(uid): int(cid) for uid, cid in zip(userids, labels)}
sc['cluster'] = sc['userid'].apply(_uid_str).map(user_cluster_map)

def _parse_genres(x):
    if isinstance(x, (list, np.ndarray)):
        return [g for g in x if g and str(g).strip()]
    if isinstance(x, str) and x.strip():
        try:
            parsed = _json.loads(x)
            if isinstance(parsed, list):
                return [g for g in parsed if g]
        except Exception:
            pass
        return [g.strip() for g in x.split(',') if g.strip()]
    return []

merged = sc.merge(
    artist_genres[['artist_name_normalized', 'genres']],
    on='artist_name_normalized', how='left'
)
merged['genres'] = merged['genres'].apply(_parse_genres)
exploded = merged.explode('genres')
exploded = exploded[exploded['genres'].notna() & (exploded['genres'].astype(str).str.strip() != '')]
exploded = exploded[exploded['cluster'].notna()]
exploded['cluster'] = exploded['cluster'].astype(int)

genre_counts = (
    exploded.groupby(['cluster', 'genres'])
    .size().reset_index(name='plays')
)
cluster_totals = genre_counts.groupby('cluster')['plays'].transform('sum')
genre_counts['pct'] = genre_counts['plays'] / cluster_totals * 100

# ── Print percentage table ────────────────────────────────────────────────────
top_genres = (
    genre_counts.groupby('genres')['plays'].sum()
    .sort_values(ascending=False).head(20).index.tolist()
)
pivot = genre_counts.pivot(index='genres', columns='cluster', values='pct').fillna(0)
pivot = pivot.loc[[g for g in top_genres if g in pivot.index]]
pivot.columns = [cluster_names.get(c, f'Cluster {c}') for c in pivot.columns]

print('Genre Breakdown By Listener Segment (% of plays within each segment)\n')
col_w = 20
print(f"{'Genre':<28}", end='')
for col in pivot.columns:
    print(f'{col[:col_w]:>{col_w}}', end='')
print()
print('-' * (28 + col_w * len(pivot.columns)))
for genre, row in pivot.iterrows():
    print(f'{genre:<28}', end='')
    for val in row:
        print(f'{val:>{col_w-1}.1f}%', end=' ')
    print()
print()

# ── 5a. Stacked bar: genre proportions across all segments ───────────────────
fig_genre = plot_genre_distribution(
    scrobbles=scrobbles,
    artist_genres=artist_genres,
    labels=labels,
    userids=userids,
    top_n_genres=15,
    cluster_names=cluster_names,
)
save_figure(fig_genre, '../outputs/figures/genre_distribution')
fig_genre.show()

# ── 5b. Per-segment genre profile: top-10 genres as horizontal bars ──────────
import plotly.graph_objects as go
from plotly.subplots import make_subplots

cluster_ids = sorted(c for c in set(labels) if c != -1)
n_clusters  = len(cluster_ids)
fig_per = make_subplots(
    rows=1, cols=n_clusters,
    subplot_titles=[cluster_names.get(c, f'Cluster {c}') for c in cluster_ids],
    shared_xaxes=True,
)

for col_idx, cid in enumerate(cluster_ids, start=1):
    seg_data = genre_counts[genre_counts['cluster'] == cid].nlargest(10, 'pct')
    fig_per.add_trace(
        go.Bar(
            x=seg_data['pct'],
            y=seg_data['genres'],
            orientation='h',
            showlegend=False,
            marker_color='#636EFA',
        ),
        row=1, col=col_idx,
    )

fig_per.update_layout(
    title='Top 10 Genres Per Listener Segment (% Of Plays)',
    template='plotly_white',
    height=420,
)
fig_per.update_xaxes(title_text='% of plays')
save_figure(fig_per, '../outputs/figures/genre_per_segment')
fig_per.show()


## 6. Temporal Listening Patterns (per segment)

In [ ]:
from src.visualization.plots import plot_temporal_heatmap, plot_temporal_ratios, plot_hour_distribution
import importlib, src.visualization.plots as _plots_mod
importlib.reload(_plots_mod)
from src.visualization.plots import plot_temporal_heatmap, plot_temporal_ratios, plot_hour_distribution

from src.visualization.plots import _uid_str
import pandas as pd, numpy as np

_user_cluster = {_uid_str(uid): int(cid) for uid, cid in zip(userids, labels)}
_sc = scrobbles.copy()
_sc['cluster'] = _sc['userid'].apply(_uid_str).map(_user_cluster)
_sc['hour'] = pd.to_datetime(_sc['timestamp']).dt.hour
_sc['dow']  = pd.to_datetime(_sc['timestamp']).dt.dayofweek

cluster_ids = sorted(c for c in set(labels) if c != -1)
global_zmax = 0.01
for _cid in cluster_ids:
    _n = sum(1 for v in _user_cluster.values() if v == _cid)
    _raw = (
        _sc[_sc['cluster'] == _cid]
        .groupby(['dow', 'hour']).size().reset_index(name='plays')
        .pivot(index='dow', columns='hour', values='plays')
        .reindex(index=range(7), columns=range(24)).fillna(0)
    ) / max(_n, 1)
    global_zmax = max(global_zmax, float(_raw.values.max()))

print(f'Shared colour-scale ceiling: {global_zmax:.3f} avg plays / user')
print(f'Scrobbles mapped to a cluster: {_sc["cluster"].notna().sum():,} / {len(_sc):,}')
print('Per-cluster scrobble counts:')
print(_sc['cluster'].value_counts(dropna=False).sort_index())

# --- 6a. Hour-of-day line chart (all segments on one chart) ------------------
fig_hour = plot_hour_distribution(
    scrobbles=scrobbles,
    labels=labels,
    userids=userids,
    cluster_names=cluster_names,
)
save_figure(fig_hour, '../outputs/figures/hour_distribution')
fig_hour.show()

# --- 6b. Temporal ratio bar chart (morning/evening/weekend by segment) -------
fig_tratio = plot_temporal_ratios(
    feature_matrix=feature_matrix,
    labels=labels,
    cluster_names=cluster_names,
)
save_figure(fig_tratio, '../outputs/figures/temporal_ratios')
fig_tratio.show()

# --- 6c. Hour x Day-of-week heatmap per segment ------------------------------
for cid in cluster_ids:
    fig_temp = plot_temporal_heatmap(
        scrobbles=scrobbles,
        labels=labels,
        userids=userids,
        cluster_id=cid,
        cluster_name=cluster_names.get(cid, f'Cluster {cid}'),
        zmax=global_zmax,
    )
    save_figure(fig_temp, f'../outputs/figures/temporal_heatmap_cluster_{cid}')
    fig_temp.show()


## 7. Business Insights & Personalization Recommendations

The cell below prints a structured summary of each segment's characteristics and actionable implications for streaming platforms.

In [ ]:
from src.clustering.evaluation import summarise_clusters, label_clusters

cluster_summary = summarise_clusters(feature_matrix, labels)
cluster_names_auto = label_clusters(cluster_summary, feature_matrix, labels)

# Key feature means per cluster (unscaled)
fm = feature_matrix.copy()
fm['cluster'] = labels

insight_features = [
    'artist_entropy', 'genre_entropy', 'unique_artists',
    'track_replay_rate', 'novelty_ratio', 'discovery_velocity_30d',
    'avg_tracks_per_session', 'weekend_ratio', 'temporal_hour_entropy',
]
insight_features = [f for f in insight_features if f in fm.columns]

global_means = fm[insight_features].mean()

print('=' * 72)
print('LISTENER SEGMENT PROFILES & PERSONALIZATION RECOMMENDATIONS')
print('=' * 72)

for cid in sorted(set(labels)):
    if cid == -1:
        continue
    cluster_users = fm[fm['cluster'] == cid]
    n = len(cluster_users)
    name = cluster_names.get(cid, f'Cluster {cid}')
    means = cluster_users[insight_features].mean()

    print(f'\nSEGMENT {cid}: {name.upper()}  ({n} users, {n/len(fm)*100:.1f}%)')
    print('-' * 60)

    for feat in insight_features:
        val = means[feat]
        gval = global_means[feat]
        delta = (val - gval) / (gval + 1e-9)
        arrow = '▲' if delta > 0.15 else ('▼' if delta < -0.15 else '—')
        print(f'  {feat:<35} {val:8.3f}  {arrow} (global: {gval:.3f})')

    # Heuristic recommendations
    recs = []
    if means.get('novelty_ratio', 0) > global_means.get('novelty_ratio', 0):
        recs.append('→ Prioritise new artist recommendations and discovery playlists')
    else:
        recs.append('→ Emphasise "More like your favourites" and artist radio')

    if means.get('track_replay_rate', 0) > global_means.get('track_replay_rate', 0) * 1.2:
        recs.append('→ Offer offline mode / download prompts for favourite tracks')

    if means.get('genre_entropy', 0) > global_means.get('genre_entropy', 0):
        recs.append('→ Cross-genre mood playlists and genre-blend features work well')
    else:
        recs.append('→ Deep-dive genre playlists and artist discography features')

    if means.get('weekend_ratio', 0) > 0.45:
        recs.append('→ Target weekend push notifications and curated weekend playlists')

    if means.get('avg_tracks_per_session', 0) > global_means.get('avg_tracks_per_session', 0) * 1.3:
        recs.append('→ Long-session features: auto-queuing, seamless transitions, sleep timer')

    print('\n  PLATFORM RECOMMENDATIONS:')
    for r in recs:
        print(f'  {r}')

print('\n' + '=' * 72)

## 8. Demographic Cross-Tabulation (Descriptive Only)

In [ ]:
def _norm_uid(uid) -> str:
    """Normalise user ID to plain integer string (handles int64, float64, str)."""
    s = str(uid)
    return s[:-2] if s.endswith(".0") else s

demo = profiles.copy()
demo['userid'] = demo['userid'].apply(_norm_uid)
demo = demo.set_index('userid')[['gender', 'age', 'country']]

ldf = labels_df.copy()
ldf['userid'] = ldf['userid'].apply(_norm_uid)
labeled = ldf.set_index('userid').join(demo)

print('Gender distribution per segment:')
print(labeled.groupby('cluster_name')['gender'].value_counts(normalize=True).round(3))

print('\nMedian age per segment:')
print(labeled.groupby('cluster_name')['age'].median())

print('\nTop 3 countries per segment:')
print(
    labeled.groupby('cluster_name')['country']
    .value_counts()
    .groupby(level=0)
    .head(3)
)

## 9. Statistical Significance — Kruskal-Wallis Tests

Non-parametric test that a feature's distribution differs across at least one pair of clusters.
Sorted by H-statistic (largest = most discriminating). Significance: `***` p<0.001 · `**` p<0.01 · `*` p<0.05 · `ns` p≥0.05.

In [ ]:
from scipy import stats

kw_features = [
    'artist_entropy', 'genre_entropy', 'unique_artists', 'artist_concentration_20',
    'track_replay_rate', 'novelty_ratio', 'discovery_velocity_30d',
    'avg_tracks_per_session', 'avg_session_length_min', 'weekend_ratio',
    'temporal_hour_entropy', 'morning_ratio', 'evening_ratio',
    'mean_energy', 'mean_valence', 'mean_danceability', 'mean_acousticness',
    'mean_tempo', 'mean_instrumentalness', 'mean_liveness', 'mean_speechiness',
]
kw_features = [f for f in kw_features if f in feature_matrix.columns]

fm_kw = feature_matrix.copy()
fm_kw['cluster'] = labels
cluster_ids = sorted(c for c in set(labels) if c != -1)

rows = []
for feat in kw_features:
    groups = [
        fm_kw.loc[fm_kw['cluster'] == cid, feat].dropna().values
        for cid in cluster_ids
    ]
    groups = [g for g in groups if len(g) > 0]
    if len(groups) < 2:
        continue
    h_stat, p_val = stats.kruskal(*groups)
    sig = '***' if p_val < 0.001 else ('**' if p_val < 0.01 else ('*' if p_val < 0.05 else 'ns'))
    rows.append({'Feature': feat, 'H': round(h_stat, 2), 'p-value': p_val, 'Sig': sig})

kw_df = pd.DataFrame(rows).sort_values('H', ascending=False).reset_index(drop=True)
kw_df['p-value'] = kw_df['p-value'].map(lambda p: f'{p:.2e}')

print('Kruskal-Wallis results — features ranked by H-statistic')
print('=' * 62)
print(kw_df.to_string(index=False))